In [23]:
!nvidia-smi
!mkdir -p /content/CUDAFUNCTIONS/include
!mkdir -p /content/CUDAFUNCTIONS/src
!mkdir -p /content/CUDAFUNCTIONS/python/ctypes
!mkdir -p /content/CUDAFUNCTIONS/python/wrapper
!mkdir -p /content/CUDAFUNCTIONS/python/wrapper_np

Fri Jun 19 07:53:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
%%writefile /content/CUDAFUNCTIONS/CMakeLists.txt
cmake_minimum_required(VERSION 3.10)
project(exposing_functions LANGUAGES CXX CUDA)

add_library(vector_add SHARED src/vector_add.cu)

set_target_properties(
    vector_add
    PROPERTIES CUDA_SEPARABLE_COMPILATION ON
)

target_include_directories(vector_add PUBLIC include)

Overwriting /content/CUDAFUNCTIONS/CMakeLists.txt


In [25]:
%%writefile /content/CUDAFUNCTIONS/include/vector_add.h
#ifndef VEC_ADD_H
#define VEC_ADD_H

extern "C" {
    void vectorAdd(int *a, int *b, int *c, int N);
}

#endif

Overwriting /content/CUDAFUNCTIONS/include/vector_add.h


In [26]:
%%writefile /content/CUDAFUNCTIONS/src/vector_add.cu
#include <cuda_runtime.h>
#include "../include/vector_add.h"

__global__ void vectorAddKernel(int *a, int *b, int *c, int N) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;

    if (idx < N) {
        c[idx] = a[idx] + b[idx];
    }
}

void vectorAdd(int *a, int *b, int *c, int N) {
    int *d_a, *d_b, *d_c;

    cudaMalloc((void**)&d_a, N * sizeof(int));
    cudaMalloc((void**)&d_b, N * sizeof(int));
    cudaMalloc((void**)&d_c, N * sizeof(int));

    cudaMemcpy(d_a, a, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, N * sizeof(int), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    vectorAddKernel<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, N);

    cudaMemcpy(c, d_c, N * sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
}

Overwriting /content/CUDAFUNCTIONS/src/vector_add.cu


In [38]:
%cd /content/CUDAFUNCTIONS
!rm -rf build
!mkdir build
%cd build
!cmake ..
!make

/content/CUDAFUNCTIONS
/content/CUDAFUNCTIONS/build
-- The CXX compiler identification is GNU 11.4.0
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Configuring done (2.5s)
CMake Warning (dev) in CMakeLists.txt:
  Policy CMP0104 is not set: CMAKE_CUDA_ARCHITECTURES now detected for NVCC,
  empty CUDA_ARCHITECTURES not allowed.  Run "cmake --help-policy CMP0104"
  for policy details.  Use the cmake_policy command to set the policy and
  suppress this warning.

  CUDA_ARCHITECTURES is empty for target "vector_add".
This 

In [39]:
%%writefile /content/CUDAFUNCTIONS/python/ctypes/test_ctypes.py
import ctypes
import numpy as np

lib = ctypes.CDLL("/content/CUDAFUNCTIONS/build/libvector_add.so")

lib.vectorAdd.argtypes = [
    ctypes.POINTER(ctypes.c_int),
    ctypes.POINTER(ctypes.c_int),
    ctypes.POINTER(ctypes.c_int),
    ctypes.c_int
]

lib.vectorAdd.restype = None

N = 10

a = np.arange(N, dtype=np.int32)
b = np.arange(N, 2 * N, dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

lib.vectorAdd(
    a.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    b.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    c.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    N
)

print("Method 1: ctypes")
print("Input A:", a)
print("Input B:", b)
print("Result :", c)

Overwriting /content/CUDAFUNCTIONS/python/ctypes/test_ctypes.py


In [44]:
%cd /content/CUDAFUNCTIONS/python/ctypes
!python3 test_ctypes.py

/content/CUDAFUNCTIONS/python/ctypes
Method 1: ctypes
Input A: [0 1 2 3 4 5 6 7 8 9]
Input B: [10 11 12 13 14 15 16 17 18 19]
Result : [10 12 14 16 18 20 22 24 26 28]


In [41]:
%%writefile /content/CUDAFUNCTIONS/python/wrapper/vector_add_wrapper.c
#define PY_SSIZE_T_CLEAN
#include <Python.h>
#include <dlfcn.h>
#include <stdlib.h>

typedef void (*vectorAddFunc)(int *, int *, int *, int);

static vectorAddFunc vectorAdd = NULL;

static PyObject* pyVectorAdd(PyObject* self, PyObject* args) {
    PyObject *py_a, *py_b;
    int N;

    if (!PyArg_ParseTuple(args, "OOi", &py_a, &py_b, &N))
        return NULL;

    if (!PyList_Check(py_a) || !PyList_Check(py_b) ||
        PyList_Size(py_a) != N || PyList_Size(py_b) != N) {
        PyErr_SetString(PyExc_ValueError, "Arguments must be lists of size N.");
        return NULL;
    }

    int *a = malloc(N * sizeof(int));
    int *b = malloc(N * sizeof(int));
    int *c = malloc(N * sizeof(int));

    for (int i = 0; i < N; i++) {
        a[i] = (int)PyLong_AsLong(PyList_GetItem(py_a, i));
        b[i] = (int)PyLong_AsLong(PyList_GetItem(py_b, i));
    }

    vectorAdd(a, b, c, N);

    PyObject *result = PyList_New(N);

    for (int i = 0; i < N; i++) {
        PyList_SetItem(result, i, PyLong_FromLong(c[i]));
    }

    free(a);
    free(b);
    free(c);

    return result;
}

static PyMethodDef VectorAddMethods[] = {
    {"vectorAdd", pyVectorAdd, METH_VARARGS, "Run CUDA vector addition"},
    {NULL, NULL, 0, NULL}
};

static struct PyModuleDef vectoraddmodule = {
    PyModuleDef_HEAD_INIT,
    "vector_add_wrapper",
    NULL,
    -1,
    VectorAddMethods
};

PyMODINIT_FUNC PyInit_vector_add_wrapper(void) {
    void *handle = dlopen(
        "/content/CUDAFUNCTIONS/build/libvector_add.so",
        RTLD_LAZY
    );

    if (!handle) {
        PyErr_SetString(PyExc_ImportError, dlerror());
        return NULL;
    }

    vectorAdd = (vectorAddFunc)dlsym(handle, "vectorAdd");

    if (!vectorAdd) {
        PyErr_SetString(PyExc_ImportError, dlerror());
        return NULL;
    }

    return PyModule_Create(&vectoraddmodule);
}

Overwriting /content/CUDAFUNCTIONS/python/wrapper/vector_add_wrapper.c


In [31]:
%%writefile /content/CUDAFUNCTIONS/python/wrapper/setup.py
from setuptools import setup, Extension

module = Extension(
    "vector_add_wrapper",
    sources=["vector_add_wrapper.c"],
    libraries=["dl"]
)

setup(
    name="vector_add_wrapper",
    version="1.0",
    ext_modules=[module]
)

Overwriting /content/CUDAFUNCTIONS/python/wrapper/setup.py


In [32]:
%%writefile /content/CUDAFUNCTIONS/python/wrapper/test_vector_add_wrapper.py
import vector_add_wrapper as vaw

N = 10

a = list(range(N))
b = list(range(N, 2 * N))

result = vaw.vectorAdd(a, b, N)

print("Method 2: Python C wrapper")
print("Input A:", a)
print("Input B:", b)
print("Result :", result)

Writing /content/CUDAFUNCTIONS/python/wrapper/test_vector_add_wrapper.py


In [33]:
%cd /content/CUDAFUNCTIONS/python/wrapper
!python3 setup.py build_ext --inplace
!python3 test_vector_add_wrapper.py

/content/CUDAFUNCTIONS/python/wrapper
running build_ext
building 'vector_add_wrapper' extension
creating build/temp.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/include/python3.12 -c vector_add_wrapper.c -o build/temp.linux-x86_64-cpython-312/vector_add_wrapper.o
creating build/lib.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/vector_add_wrapper.o -L/usr/lib/x86_64-linux-gnu -ldl -o build/lib.linux-x86_64-cpython-312/vector_add_wrapper.cpython-312-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-312/vector_add_wrapper.cpython-312-x86_64-linux-gnu.so -> 
Method 2: Python C wrapper
Input A: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Input B: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Result : [10, 12, 14, 16, 18, 20, 22, 24, 26, 28

In [34]:
%%writefile /content/CUDAFUNCTIONS/python/wrapper_np/vector_add_np_wrapper.c
#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION
#define PY_SSIZE_T_CLEAN

#include <Python.h>
#include <numpy/arrayobject.h>
#include <dlfcn.h>

typedef void (*vectorAddFunc)(int *, int *, int *, int);

static vectorAddFunc vectorAdd = NULL;

static PyObject* pyVectorAdd(PyObject* self, PyObject* args) {
    PyArrayObject *a;
    PyArrayObject *b;
    PyArrayObject *c;
    int N;

    if (!PyArg_ParseTuple(
        args,
        "O!O!O!i",
        &PyArray_Type, &a,
        &PyArray_Type, &b,
        &PyArray_Type, &c,
        &N
    )) {
        return NULL;
    }

    if (PyArray_TYPE(a) != NPY_INT32 ||
        PyArray_TYPE(b) != NPY_INT32 ||
        PyArray_TYPE(c) != NPY_INT32) {
        PyErr_SetString(PyExc_TypeError, "Arrays must use dtype int32.");
        return NULL;
    }

    if (PyArray_SIZE(a) != N ||
        PyArray_SIZE(b) != N ||
        PyArray_SIZE(c) != N) {
        PyErr_SetString(PyExc_ValueError, "Arrays must have size N.");
        return NULL;
    }

    int *a_ptr = (int*)PyArray_DATA(a);
    int *b_ptr = (int*)PyArray_DATA(b);
    int *c_ptr = (int*)PyArray_DATA(c);

    vectorAdd(a_ptr, b_ptr, c_ptr, N);

    Py_RETURN_NONE;
}

static PyMethodDef VectorAddMethods[] = {
    {"vectorAdd", pyVectorAdd, METH_VARARGS, "Run CUDA vector addition with NumPy"},
    {NULL, NULL, 0, NULL}
};

static struct PyModuleDef vectoraddmodule = {
    PyModuleDef_HEAD_INIT,
    "vector_add_np_wrapper",
    NULL,
    -1,
    VectorAddMethods
};

PyMODINIT_FUNC PyInit_vector_add_np_wrapper(void) {
    void *handle = dlopen(
        "/content/CUDAFUNCTIONS/build/libvector_add.so",
        RTLD_LAZY
    );

    if (!handle) {
        PyErr_SetString(PyExc_ImportError, dlerror());
        return NULL;
    }

    vectorAdd = (vectorAddFunc)dlsym(handle, "vectorAdd");

    if (!vectorAdd) {
        PyErr_SetString(PyExc_ImportError, dlerror());
        return NULL;
    }

    import_array();

    return PyModule_Create(&vectoraddmodule);
}

Writing /content/CUDAFUNCTIONS/python/wrapper_np/vector_add_np_wrapper.c


In [35]:
%%writefile /content/CUDAFUNCTIONS/python/wrapper_np/setup.py
from setuptools import setup, Extension
import numpy

module = Extension(
    "vector_add_np_wrapper",
    sources=["vector_add_np_wrapper.c"],
    include_dirs=[numpy.get_include()],
    libraries=["dl"]
)

setup(
    name="vector_add_np_wrapper",
    version="1.0",
    ext_modules=[module]
)

Writing /content/CUDAFUNCTIONS/python/wrapper_np/setup.py


In [36]:
%%writefile /content/CUDAFUNCTIONS/python/wrapper_np/test_vector_add_np_wrapper.py
import numpy as np
import vector_add_np_wrapper as vaw_np

N = 10

a = np.arange(N, dtype=np.int32)
b = np.arange(N, 2 * N, dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

vaw_np.vectorAdd(a, b, c, N)

print("Method 3: NumPy C wrapper")
print("Input A:", a)
print("Input B:", b)
print("Result :", c)

Writing /content/CUDAFUNCTIONS/python/wrapper_np/test_vector_add_np_wrapper.py


In [37]:
%cd /content/CUDAFUNCTIONS/python/wrapper_np
!python3 setup.py build_ext --inplace
!python3 test_vector_add_np_wrapper.py

/content/CUDAFUNCTIONS/python/wrapper_np
running build_ext
building 'vector_add_np_wrapper' extension
creating build/temp.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/local/lib/python3.12/dist-packages/numpy/_core/include -I/usr/include/python3.12 -c vector_add_np_wrapper.c -o build/temp.linux-x86_64-cpython-312/vector_add_np_wrapper.o
creating build/lib.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/vector_add_np_wrapper.o -L/usr/lib/x86_64-linux-gnu -ldl -o build/lib.linux-x86_64-cpython-312/vector_add_np_wrapper.cpython-312-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-312/vector_add_np_wrapper.cpython-312-x86_64-linux-gnu.so -> 
Method 3: NumPy C wrapper
Input A: [0 1 2 3 4 5 6 7 8 9]
Input B: [10 11 12 13 14 